# 파일 입력
비전 기능을 갖춘 OpenAI 모델은 PDF 파일을 입력으로 사용할 수도 있습니다. 

모델이 PDF 콘텐츠를 이해할 수 있도록 각 페이지의 추출된 텍스트와 이미지를 모델의 컨텍스트에 추가합니다. 모델은 텍스트와 이미지를 모두 사용하여 응답을 생성할 수 있습니다. 이는 예를 들어 다이어그램에 텍스트에 없는 주요 정보가 포함된 경우 유용합니다.

먼저 Files API를 사용하여 PDF를 업로드한 다음, 모델에 대한 API 요청에서 해당 파일 ID를 참조합니다.

In [1]:
from dotenv import load_dotenv
load_dotenv()

True

In [2]:
from openai import OpenAI

client = OpenAI()

Model = "gpt-5-nano"

## 파일 URL (File URLs) 사용

PDF 파일을 직접 업로드하지 않고, 공개된 외부 PDF 링크(URL)를 통해 모델이 해당 파일에 접근하도록 설정할 수 있습니다.

In [7]:
from openai import OpenAI
client = OpenAI()

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_text",
                    "text": "편지를 분석하고 핵심 요점을 요약해 주세요",
                },
                {
                    "type": "input_file",
                    "file_url": "https://www.berkshirehathaway.com/letters/2024ltr.pdf",
                },
            ],
        },
    ]
)

print(response.output_text)

다음은 Berkshire Hathaway의 버핏 편지에서 제시된 핵심 요점의 간략 요약입니다.

- 편지의 의의
  - 연차보고서의 데이터 제시뿐 아니라 소유자 시각에서의 해설과 생각을 전하려는 의도.
  - 문제점은 솔직히 인정하고, 필요한 경우 적극적으로 바로잡아야 한다는 원칙 강조.

- 실수와 학습
  - 과거 사업성 평가나 경영자 평가의 실수가 있었고, 이를 바로잡는 것이 중요하다고 강조.
  - “자신의 baloney를 믿지 말고, 필요한 시점에 행동하라”는 교훈(고객·주주 입장에서의 책임).

- Pete Liegl 이야기
  - Forest River 창업주 Pete Liegl의 사례를 통해 비전과 실행력, 관리자의 질이 얼마나 큰 차이를 만드는지 설명.
  - 학교 배경보다 타고난 재능과 꾸준한 성과의 중요성을 강조.

- 2024년 성과 요약
  - 연간 영업이익: 474억 달러.
  - 손익 구분: 보험인수익 90.2억 달러, 보험투자수익 136.7억 달러, BNSF 50.31억 달러, Berkshire Hathaway Energy 37.3억 달러 등.
  - 비(非)통제 사업 포함 기타 부문 등 상세 분해 제공.
  - 주가가 아닌 영업현금흐름 및 장기 관점의 성과 지표 강조(가치창출은 수년, 수십년 단위로 본다).

- 절대적 메시지: 현금보다 주식에 투자하는 전략
  - 회사의 현금 보유고보다 대부분의 자금을 주식(특히 미국 기업의 고질적 가치주)과 통제/비통제 기업에 배분하겠다는 기본 전략 재확인.
  - 현금이나 채권 등 금융상품보다 고품질 비즈니스에 대한 소유권이 더 큰 가치 창출 원천이라는 주장.

- 자본배분의 양면성: 통제 vs 비통제
  - 통제기업은 경영·자본흐름에 대한 결정력 있지만 매매의 유연성은 감소.
  - 비통제 주식은 관리 교체나 지배력 행사에 제약이 있어, 실수의 바로잡기가 어렵다는 점도 인식.

- 보험(P/C) 비즈니스의 핵심과 리스크
  - Berkshire의 핵심이 여전히 P/C 보험.
  - 선지급-

### 파일 업로드 (Uploading files)

먼저 Files API를 사용하여 PDF 파일을 업로드한 다음, 해당 파일의 파일 ID(file ID) 를 모델의 API 요청에서 참조

In [4]:
file = client.files.create(
    file=open("data/삼성중공업-기업리포트.pdf", "rb"),
    purpose="user_data"
)

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "file_id": file.id,
                },
                {
                    "type": "input_text",
                    "text": "파일 내에 있는 첫번째 바차트를 설명해줘.",
                },
            ]
        }
    ]
)

print(response.output_text)

다음 파일의 첫 번째 바 차트는 “주가추이 및 상대강도”입니다. 삼성중공업 주가의 절대 수익률과 KOSPI 대비 상대 수익률을 1개월/3개월/6개월 시점으로 비교해 보여줍니다.

데이터 포인트
- 1개월(M): 절대수익률 7.8%, 상대수익률 1.8%
- 3개월(M): 절대수익률 15.7%, 상대수익률 8.3%
- 6개월(M): 절대수익률 38.9%, 상대수익률 28.1%

해석
- 각 기간마다 두 개의 막대가 표시되어, 주가의 자체 상승폭(절대 수익률)과 시장 대비 성과(상대 수익률)가 함께 나타납니다.
- 6개월 기준 절대 수익률이 38.9%로 큰 폭 상승했고, 상대 수익률도 28.1%로 KOSPI 대비 상당히 강한 아웃퍼폼을 보였습니다. 이는 최근 모멘텀이나 실적 개선에 의한 긍정적 흐름을 시사합니다.


### Base64로 인코딩된 파일
PDF 파일 입력도 Base64로 인코딩된 입력으로 보낼 수 있습니다.

In [5]:
import base64

with open("data/삼성중공업-기업리포트.pdf", "rb") as f:
    data = f.read()

base64_string = base64.b64encode(data).decode("utf-8")

response = client.responses.create(
    model=Model,
    input=[
        {
            "role": "user",
            "content": [
                {
                    "type": "input_file",
                    "filename": "삼성중공업-기업리포트.pdf",
                    "file_data": f"data:application/pdf;base64,{base64_string}",
                },
                {
                    "type": "input_text",
                    "text": "파일 내에 있는 첫번째 바차트를 설명해줘.",
                },
            ],
        },
    ]
)

print(response.output_text)

다음이 파일의 첫 번째 바 차트에 대한 설명입니다.

- 차트 주제: 삼성중공업의 주가 흐름과 상대강도 비교
- 무엇을 비교하나: 1개월(1M), 3개월(3M), 6개월(6M) 동안의 절대수익률과 상대수익률을 한 눈에 비교
- 축과 구성: 
  - x축: 기간(1M, 3M, 6M)
  - 각 기간마다 두 개의 바가 표시되며, 하나는 절대수익률, 다른 하나는 상대수익률을 나타냄
- 수치(차트에 표시된 값):
  - 1M: 절대수익률 7.8%, 상대수익률 1.8%
  - 3M: 절대수익률 15.7%, 상대수익률 8.3%
  - 6M: 절대수익률 38.9%, 상대수익률 28.1%
- 해석 포인트:
  - 6개월 기간 동안 주가의 상승 폭이 크고, 시장 대비 초과수익도 나타나고 있음(상대수익률이 양의 값).
  - 절대수익률이 상대수익률보다 큰 경우가 많아, 전체적으로 주가 상승이 뚜렷하나, 시장 대비 상대적 초과수익은 기간에 따라 차이가 있음.

추가로 다른 차트나 세부 설명이 필요하면 말씀해 주세요.


---------------------------------
### 사용 시 고려사항 (Usage considerations)

- 토큰 사용량 (Token usage)

    모델이 PDF 내용을 이해할 수 있도록, OpenAI는 각 페이지의 추출된 텍스트와 페이지 이미지를 모두 모델의 컨텍스트에 포함시킵니다. 따라서 PDF를 입력으로 사용할 경우 토큰 사용량과 비용이 증가할 수 있습니다.

- 파일 크기 제한 (File size limitations)

    한 번에 여러 개의 파일을 업로드할 수 있습니다. 단, 각 파일은 50MB 미만이어야 합니다.  
    하나의 API 요청 내에서 전체 파일 용량의 합도 50MB를 초과할 수 없습니다.